In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/isic-flagship-project")
os.chdir(PROJECT_ROOT)

print("Working in:", PROJECT_ROOT)

Working in: /content/drive/MyDrive/isic-flagship-project


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/services/copilot_support_service.py
"""
Support service for the Copilot Studio agent.

Keep this service lightweight. Cloud Run should not load embeddings or build
a vector index during API startup.
"""

from typing import Dict, List


class CopilotSupportService:
    def __init__(self):
        self.medical_terms = [
            "diagnose",
            "diagnosis",
            "cancer",
            "melanoma",
            "benign",
            "malignant",
            "treatment",
            "treat",
            "medicine",
            "should i see",
            "what is this lesion",
            "is this lesion",
            "is this mole",
            "is this dangerous",
            "do i have",
            "am i sick",
            "what should i do medically",
        ]

    def answer_question(self, question: str) -> Dict:
        clean_question = question.strip()
        intent = self._classify_intent(clean_question)

        if intent == "medical_advice":
            return self._medical_refusal()

        return {
            "answer": self._build_answer(intent),
            "intent": intent,
            "risk_level": self._risk_level_for_intent(intent),
            "automation_allowed": True,
            "escalation_required": False,
            "sources": self._sources_for_intent(intent),
            "safety_note": (
                "This agent provides technical support for the platform. "
                "It does not provide medical diagnosis, treatment advice, "
                "or clinical interpretation of uploaded images."
            ),
        }

    def _classify_intent(self, question: str) -> str:
        q = question.lower()

        if any(term in q for term in self.medical_terms):
            return "medical_advice"

        if any(term in q for term in ["failed", "fail", "error", "troubleshoot", "400", "500"]):
            return "failed_upload_support"

        if any(term in q for term in ["upload", "image file", "file type", "multipart"]):
            return "image_upload_support"

        if any(term in q for term in ["endpoint", "api", "predict", "/predict", "request", "response"]):
            return "api_support"

        if any(term in q for term in ["probability", "score", "confidence", "risk score"]):
            return "prediction_explanation"

        if any(term in q for term in ["govern", "governance", "safety", "limitation", "human review", "audit"]):
            return "governance"

        return "general_platform_support"

    def _build_answer(self, intent: str) -> str:
        answers = {
            "api_support": (
                "The prediction endpoint accepts an uploaded image, validates the file, "
                "passes it through the inference pipeline, and returns a structured prediction response.\n\n"
                "A typical response includes the uploaded image identifier, the model probability, "
                "the prediction label, and model/version metadata used for traceability.\n\n"
                "The endpoint is intended for platform workflow support and risk review. "
                "Its output should not be treated as a medical diagnosis."
            ),
            "image_upload_support": (
                "To upload an image, the client should send the file to the prediction endpoint "
                "as a multipart/form-data request.\n\n"
                "The platform checks that the uploaded file is an image before sending it to the "
                "prediction service. If the file is not recognised as an image, the API should reject "
                "the request instead of passing it into the model.\n\n"
                "For Copilot Studio users, this should be explained as a technical upload process, "
                "not as a clinical image review."
            ),
            "failed_upload_support": (
                "A failed upload is usually caused by one of these issues:\n\n"
                "- the file was not sent as multipart/form-data\n"
                "- the uploaded file is not a supported image type\n"
                "- the request used the wrong endpoint URL\n"
                "- the file field name does not match the API contract\n"
                "- the backend service is unavailable or returned an internal error\n\n"
                "Check the request format first, then confirm the endpoint health, and finally "
                "review the API logs for the exact failure reason."
            ),
            "prediction_explanation": (
                "The probability value is the model's numerical output for the prediction request. "
                "It should be read as a model-generated risk signal, not as a diagnosis.\n\n"
                "A higher probability can be used by the platform to trigger review workflows, "
                "such as human-in-the-loop review, but it should not be used to tell a user that "
                "they do or do not have a medical condition.\n\n"
                "The correct framing is: probability supports workflow prioritisation and review, "
                "not clinical decision-making."
            ),
            "governance": (
                "The model is governed through safety rules, action tiers, human review, and clear "
                "limits on what the system is allowed to automate.\n\n"
                "Low-risk actions, such as explaining API usage or upload steps, can be automated. "
                "Medium-risk actions, such as flagging uncertain predictions, should route to human "
                "review. High-risk or prohibited actions, such as diagnosing a lesion or recommending "
                "treatment, must not be automated.\n\n"
                "The agent should always make the medical safety boundary clear."
            ),
            "general_platform_support": (
                "I can help with technical questions about the Skin Lesion Platform, including the "
                "prediction endpoint, image upload process, probability score, failed uploads, "
                "workflow review, governance, and safety limitations.\n\n"
                "I cannot provide diagnosis, treatment advice, or clinical interpretation of skin lesions."
            ),
        }

        return answers.get(intent, answers["general_platform_support"])

    def _sources_for_intent(self, intent: str) -> List[str]:
        source_map = {
            "api_support": [
                "src/api/routes.py",
                "src/services/prediction_service.py",
                "docs/api.md",
            ],
            "image_upload_support": [
                "src/api/routes.py",
                "docs/api.md",
            ],
            "failed_upload_support": [
                "src/api/routes.py",
                "src/services/prediction_service.py",
                "docs/api.md",
            ],
            "prediction_explanation": [
                "src/schemas/prediction.py",
                "governance/medical_ai_safety_policy.md",
            ],
            "governance": [
                "governance/action_tier_model.md",
                "governance/classification_canon.md",
                "governance/human_in_the_loop_policy.md",
                "governance/medical_ai_safety_policy.md",
            ],
            "general_platform_support": [
                "README.md",
                "copilot_studio/README.md",
            ],
        }

        return source_map.get(intent, source_map["general_platform_support"])

    def _risk_level_for_intent(self, intent: str) -> str:
        if intent in {"prediction_explanation", "governance"}:
            return "Medium"

        return "Low"

    def _medical_refusal(self) -> Dict:
        return {
            "answer": (
                "I can help with technical questions about the Skin Lesion Platform, including "
                "the API, upload flow, prediction response format, governance process, and safety controls.\n\n"
                "I cannot interpret a lesion, confirm whether it is cancer, decide whether it is benign "
                "or malignant, or recommend treatment. For medical concerns, please speak with a "
                "qualified clinician."
            ),
            "intent": "medical_advice",
            "risk_level": "Prohibited",
            "automation_allowed": False,
            "escalation_required": True,
            "sources": [
                "governance/medical_ai_safety_policy.md",
                "governance/action_tier_model.md",
                "governance/human_in_the_loop_policy.md",
            ],
            "safety_note": (
                "Medical diagnosis, lesion interpretation, and treatment advice are outside "
                "the agent's allowed scope."
            ),
        }

Overwriting /content/drive/MyDrive/isic-flagship-project/src/services/copilot_support_service.py


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/copilot_routes.py
"""
Routes used by Copilot Studio.
"""

from fastapi import APIRouter

from src.schemas.copilot_agent import CopilotSupportRequest, CopilotSupportResponse
from src.services.copilot_support_service import CopilotSupportService

copilot_router = APIRouter()


def get_support_service() -> CopilotSupportService:
    return CopilotSupportService()


@copilot_router.get("/agent/health")
async def copilot_agent_health():
    return {
        "status": "ok",
        "agent": "Skin Lesion Platform Support Agent",
        "scope": "technical_support_only",
    }


@copilot_router.post("/agent/support", response_model=CopilotSupportResponse)
async def ask_support_agent(request: CopilotSupportRequest):
    service = get_support_service()
    result = service.answer_question(request.question)
    return CopilotSupportResponse(**result)

Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/copilot_routes.py


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/main.py
"""
Cloud Run entry point for the Copilot Studio support API.
"""

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from src.api.copilot_routes import copilot_router


app = FastAPI(
    title="ISIC Skin Lesion Platform Support API",
    version="1.0.0",
    description="Lightweight API used by the Copilot Studio support agent.",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
async def root():
    return {
        "status": "ok",
        "service": "ISIC Copilot Support API",
    }


@app.get("/api/v1/health")
async def health():
    return {
        "status": "ok",
        "service": "copilot-support-api",
    }


app.include_router(
    copilot_router,
    prefix="/api/v1",
    tags=["copilot-studio"],
)

Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/main.py


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/isic-flagship-project")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.api.main import app

routes = sorted(route.path for route in app.routes)

print("App import OK")
print("Routes:")
for route in routes:
    print("-", route)

assert "/api/v1/agent/health" in routes
assert "/api/v1/agent/support" in routes

print("Copilot routes OK")

App import OK
Routes:
- /
- /api/v1/agent/health
- /api/v1/agent/support
- /api/v1/health
- /docs
- /docs/oauth2-redirect
- /openapi.json
- /redoc
Copilot routes OK


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/Dockerfile
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y \
    git \
    curl \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .

RUN pip install --upgrade pip

RUN pip install --no-cache-dir \
    torch==2.3.1 \
    torchvision==0.18.1 \
    --index-url https://download.pytorch.org/whl/cpu

RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY .env.example .env

EXPOSE 8080

CMD exec uvicorn src.api.main:app --host 0.0.0.0 --port ${PORT:-8080}

Overwriting /content/drive/MyDrive/isic-flagship-project/Dockerfile


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/requirements.txt
fastapi==0.115.2
uvicorn[standard]==0.32.0
pydantic==2.9.2
pydantic-settings==2.6.1
sqlalchemy==2.0.35
alembic==1.13.2
aiosqlite==0.20.0
python-dotenv==1.0.1
pandas==2.2.2
numpy==1.26.4
pillow==10.4.0
timm==1.0.11
lightgbm==4.5.0
catboost==1.2.7
xgboost==2.1.1
scikit-learn==1.5.2
langchain==0.3.4
langchain-community==0.3.3
langchain-text-splitters==0.3.0
faiss-cpu==1.8.0.post1
sentence-transformers==3.1.1
python-multipart==0.0.9
httpx==0.27.2
structlog==24.4.0

Overwriting /content/drive/MyDrive/isic-flagship-project/requirements.txt


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/.dockerignore
__pycache__/
*.pyc
*.pyo
*.pyd
.Python
env/
venv/
.venv/
.git/
.gitignore
.ipynb_checkpoints/
*.ipynb
logs/

Overwriting /content/drive/MyDrive/isic-flagship-project/.dockerignore


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
PROJECT_ID = "isic-flagship-project"
REGION = "europe-west2"
SERVICE_NAME = "isic-api"

!gcloud config set project {PROJECT_ID}

Updated property [core/project].


In [ ]:
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]

SERVICE_ACCOUNT = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"

print("Project number:", PROJECT_NUMBER)
print("Cloud Run service account:", SERVICE_ACCOUNT)

Project number: 918647643601
Cloud Run service account: 918647643601-compute@developer.gserviceaccount.com


In [ ]:
!gcloud secrets add-iam-policy-binding power-automate-url \
  --member="serviceAccount:{SERVICE_ACCOUNT}" \
  --role="roles/secretmanager.secretAccessor" \
  --project={PROJECT_ID}

Updated IAM policy for secret [power-automate-url].
bindings:
- members:
  - serviceAccount:918647643601-compute@developer.gserviceaccount.com
  role: roles/secretmanager.secretAccessor
etag: BwZSMTNL54E=
version: 1


In [ ]:
%cd /content/drive/MyDrive/isic-flagship-project

!gcloud run deploy {SERVICE_NAME} \
  --source . \
  --region {REGION} \
  --allow-unauthenticated \
  --memory 4Gi \
  --cpu 1 \
  --min-instances 1 \
  --max-instances 3 \
  --timeout 300 \
  --port 8080 \
  --cpu-boost \
  --set-secrets POWER_AUTOMATE_URL=power-automate-url:latest \
  --project {PROJECT_ID}

/content
Building using Dockerfile and deploying container to Cloud Run service [isic-api] in project [isic-flagship-project] region [europe-west2]
Service [isic-api] revision [isic-api-00005-4zj] has been deployed and is serving 100 percent of traffic.
Service URL: https://isic-api-918647643601.europe-west2.run.app


In [ ]:
import requests

CLOUD_RUN_URL = "https://isic-api-918647643601.europe-west2.run.app"

health = requests.get(f"{CLOUD_RUN_URL}/api/v1/agent/health", timeout=30)
print("Health:", health.status_code)
print(health.json())

payload = {
    "question": "How does the prediction endpoint work?",
    "conversation_id": "copilot-smoke-test-001",
    "user_role": "user",
}

support = requests.post(
    f"{CLOUD_RUN_URL}/api/v1/agent/support",
    json=payload,
    timeout=30,
)

print("Support:", support.status_code)
print(support.json())

Health: 200
{'status': 'ok', 'agent': 'Skin Lesion Platform Support Agent', 'scope': 'technical_support_only'}
Support: 200
{'answer': 'The prediction endpoint accepts an uploaded image, validates the file, passes it through the inference pipeline, and returns a structured prediction response.\n\nA typical response includes the uploaded image identifier, the model probability, the prediction label, and model/version metadata used for traceability.\n\nThe endpoint is intended for platform workflow support and risk review. Its output should not be treated as a medical diagnosis.', 'intent': 'api_support', 'risk_level': 'Low', 'automation_allowed': True, 'escalation_required': False, 'sources': ['src/api/routes.py', 'src/services/prediction_service.py', 'docs/api.md'], 'safety_note': 'This agent provides technical support for the platform. It does not provide medical diagnosis, treatment advice, or clinical interpretation of uploaded images.'}
